# Word2Vec Word Embeddings using CBOW and Skip-Gram

**Aim:** Take text sentences directly from the user, preprocess them, and train Word2Vec models using CBOW and Skip-Gram.

**Software:** Python 3, Jupyter Notebook / VS Code Jupyter, Gensim, NumPy, Pandas, Matplotlib, Scikit-learn.

**How input works:** When you run the input cell, type or paste several sentences. Press Enter after each sentence. Submit a blank line to finish entering the text. The model trains on the sentences you entered. No training sentences are hardcoded.


## 1. Theory

**Word embeddings** are dense numerical vectors that represent words. Words that occur in similar contexts tend to receive similar vector representations. Unlike one-hot encoding, embeddings use compact vectors and can capture relationships between words.

**Word2Vec** is a neural-network-based method that learns word representations from surrounding context. It has two architectures:

**CBOW (Continuous Bag of Words):** predicts a target word using its surrounding context words. Its input is context words and its output is the target word. It is generally faster to train.

**Skip-Gram:** predicts surrounding context words using a target word. Its input is the target word and its output is context words. It can work well for infrequent words, but training is generally slower.

| Feature | CBOW | Skip-Gram |
|---|---|---|
| Input | Context words | Target word |
| Prediction | Target word | Context words |
| Training speed | Usually faster | Usually slower |

Important parameters: `vector_size` is the number of dimensions in each word vector; `window` controls context distance; `min_count` ignores rare words; `sg=0` selects CBOW and `sg=1` selects Skip-Gram.

**Cosine similarity** measures the similarity between two vectors. A value closer to 1 indicates that the vectors point in similar directions.

Applications include search engines, sentiment analysis, text classification, machine translation, chatbots, and document similarity.


## 2. Install and import libraries

In [ ]:
%pip install -q gensim numpy pandas matplotlib scikit-learn

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
from sklearn.decomposition import PCA

print("Libraries imported successfully.")

## 3. Take text input from the user

Run this cell and enter multiple sentences, one sentence per line. Press **Enter on an empty line** to finish. You can paste a paragraph too, but separate sentences with punctuation for better sentence splitting.

In [ ]:
print("Enter your text below, one or more sentences.")
print("Press Enter on a blank line when you have finished.\\n")

user_lines = []

while True:
    line = input()
    if line.strip() == "":
        if user_lines:
            break
        else:
            print("Please enter at least one sentence before finishing.")
            continue
    user_lines.append(line)

user_text = " ".join(user_lines)

print("\\nText received successfully.")
print("Number of input lines:", len(user_lines))
print("Your text:")
print(user_text)

## 4. Preprocess and tokenize the user's text

The text is split into sentences and each sentence is converted into a list of lowercase tokens. Word2Vec requires a list of token lists.

In [ ]:
# Split the user-provided text into sentence-like units.
sentence_parts = re.split(r"(?<=[.!?])\\s+|\\n+", user_text)

tokenized_corpus = []
for sentence in sentence_parts:
    tokens = simple_preprocess(sentence, deacc=True, min_len=2)
    if tokens:
        tokenized_corpus.append(tokens)

if not tokenized_corpus:
    raise ValueError("No usable words found. Please enter meaningful text.")

total_words = sum(len(sentence) for sentence in tokenized_corpus)

print("Number of tokenized sentences:", len(tokenized_corpus))
print("Total words:", total_words)
print("Tokenized corpus:")
for sentence in tokenized_corpus:
    print(sentence)

if total_words < 10:
    print("\\nWarning: Your input is very short. Word2Vec works better with a larger text corpus.")

## 5. Train the CBOW Word2Vec model

In Gensim, `sg=0` selects CBOW. It learns to predict a target word from its surrounding context words.

In [ ]:
cbow_model = Word2Vec(
    sentences=tokenized_corpus,
    vector_size=100,
    window=3,
    min_count=1,
    workers=1,
    sg=0,          # 0 = CBOW
    epochs=100,
    seed=42
)

print("CBOW model training completed.")
print("CBOW vocabulary size:", len(cbow_model.wv))

## 6. Train the Skip-Gram Word2Vec model

In Gensim, `sg=1` selects Skip-Gram. It learns to predict surrounding context words from a target word.

In [ ]:
skipgram_model = Word2Vec(
    sentences=tokenized_corpus,
    vector_size=100,
    window=3,
    min_count=1,
    workers=1,
    sg=1,          # 1 = Skip-Gram
    epochs=100,
    seed=42
)

print("Skip-Gram model training completed.")
print("Skip-Gram vocabulary size:", len(skipgram_model.wv))

## 7. Display the word vector for a word entered by the user

In [ ]:
word = input("Enter a word from your text: ").strip().lower()

if word in cbow_model.wv:
    print("\\nCBOW word vector:")
    print(cbow_model.wv[word])
    print("Vector dimensions:", cbow_model.wv[word].shape)
else:
    print("Word not found in CBOW vocabulary.")

if word in skipgram_model.wv:
    print("\\nSkip-Gram word vector:")
    print(skipgram_model.wv[word])
else:
    print("Word not found in Skip-Gram vocabulary.")

## 8. Find similar words for a user-entered word

In [ ]:
word = input("Enter a word to find similar words: ").strip().lower()

if word in cbow_model.wv:
    print("\\nCBOW similar words:")
    print(cbow_model.wv.most_similar(word, topn=min(5, len(cbow_model.wv) - 1)))
else:
    print("Word not found in CBOW vocabulary.")

if word in skipgram_model.wv:
    print("\\nSkip-Gram similar words:")
    print(skipgram_model.wv.most_similar(word, topn=min(5, len(skipgram_model.wv) - 1)))
else:
    print("Word not found in Skip-Gram vocabulary.")

## 9. Calculate cosine similarity between two user-entered words

In [ ]:
word1 = input("Enter the first word: ").strip().lower()
word2 = input("Enter the second word: ").strip().lower()

if word1 in cbow_model.wv and word2 in cbow_model.wv:
    print("CBOW cosine similarity:",
          cbow_model.wv.similarity(word1, word2))
else:
    print("One or both words are not in the CBOW vocabulary.")

if word1 in skipgram_model.wv and word2 in skipgram_model.wv:
    print("Skip-Gram cosine similarity:",
          skipgram_model.wv.similarity(word1, word2))
else:
    print("One or both words are not in the Skip-Gram vocabulary.")

## 10. Compare vocabulary and vector dimensions

In [ ]:
print("CBOW vocabulary size:", len(cbow_model.wv))
print("Skip-Gram vocabulary size:", len(skipgram_model.wv))
print("CBOW vector dimensions:", cbow_model.wv.vector_size)
print("Skip-Gram vector dimensions:", skipgram_model.wv.vector_size)

## 11. Visualise embeddings using PCA

PCA reduces the word vectors to two dimensions for plotting. The graph uses the most frequent words from the user's input. A useful plot requires at least three distinct vocabulary words.

In [ ]:
words_to_plot = cbow_model.wv.index_to_key[:30]

if len(words_to_plot) >= 3:
    vectors = np.array([cbow_model.wv[word] for word in words_to_plot])
    reduced = PCA(n_components=2).fit_transform(vectors)

    plt.figure(figsize=(10, 7))
    plt.scatter(reduced[:, 0], reduced[:, 1])

    for i, word in enumerate(words_to_plot):
        plt.annotate(
            word,
            (reduced[i, 0], reduced[i, 1]),
            xytext=(4, 4),
            textcoords="offset points"
        )

    plt.title("CBOW Word Embeddings (PCA)")
    plt.xlabel("Principal Component 1")
    plt.ylabel("Principal Component 2")
    plt.grid(True)
    plt.show()
else:
    print("Enter at least three distinct words to create a PCA plot.")

## 12. Save the trained models

In [ ]:
cbow_model.save("word2vec_cbow.model")
skipgram_model.save("word2vec_skipgram.model")

print("CBOW model saved as word2vec_cbow.model")
print("Skip-Gram model saved as word2vec_skipgram.model")

## 13. Result and conclusion

**Result:** Text was entered directly by the user. The input was preprocessed and used to train CBOW and Skip-Gram Word2Vec models. The notebook displayed word vectors, found similar words, calculated cosine similarity, and visualised embeddings using PCA.

**Conclusion:** Word2Vec learns numerical representations of words from their surrounding context. CBOW predicts a target word from context words, while Skip-Gram predicts context words from a target word. The quality of the embeddings depends on the amount and quality of text entered.

## 14. Viva questions

1. **What is a word embedding?** A dense numerical vector representing a word.
2. **What is Word2Vec?** A technique that learns word vectors from surrounding context.
3. **What is CBOW?** It predicts a target word from context words.
4. **What is Skip-Gram?** It predicts context words from a target word.
5. **What is a context window?** The number of neighbouring words considered around a target word.
6. **What does `sg=0` mean?** CBOW.
7. **What does `sg=1` mean?** Skip-Gram.
8. **What is cosine similarity?** A measure of similarity between two vectors.
